# 27. Pandas Capstone Mini-Project

## 🛒 Global E-Commerce & Retail Analytics Pipeline

An end-to-end data science pipeline built exclusively with **Pandas** to ingest, clean, feature-engineer, aggregate, and visualize performance insights on a realistic retail transactions dataset.

### 1. Generating & Ingesting Dataset Matrix

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_records = 250

categories = ['Technology', 'Office Supplies', 'Furniture']
sub_cats = {'Technology': ['Phones', 'Laptops', 'Accessories'],
            'Office Supplies': ['Paper', 'Binders', 'Storage'],
            'Furniture': ['Chairs', 'Tables', 'Bookcases']}

records = []
for i in range(1, n_records + 1):
    cat = np.random.choice(categories)
    sub_cat = np.random.choice(sub_cats[cat])
    sales = np.random.randint(20, 1200)
    discount = np.random.choice([0.0, 0.05, 0.10, 0.20, 0.30])
    profit = round(sales * (np.random.uniform(0.05, 0.35) - discount), 2)
    records.append({
        'Order_ID': f"ORD-2026-{1000 + i}",
        'Order_Date': pd.Timestamp('2026-01-01') + pd.Timedelta(days=int(np.random.randint(0, 180))),
        'Customer_ID': f"CUST-{np.random.randint(100, 150)}",
        'Region': np.random.choice(['East', 'West', 'Central', 'South']),
        'Category': cat,
        'Sub_Category': sub_cat,
        'Sales': sales,
        'Quantity': np.random.randint(1, 9),
        'Discount': discount,
        'Profit': profit
    })

df_retail = pd.DataFrame(records)
# Introduce deliberate missing values for data cleaning demonstration
df_retail.loc[np.random.choice(df_retail.index, size=10, replace=False), 'Profit'] = np.nan
print("Raw Retail Matrix Head:")
print(df_retail.head())

### 2. Data Cleaning & Missing Value Treatment

In [ ]:
print(f"Initial Missing Values:\n{df_retail.isna().sum()}")

# Impute missing profits using median profit of each Sub_Category
df_retail['Profit'] = df_retail.groupby('Sub_Category')['Profit'].transform(lambda g: g.fillna(g.median()))
print(f"\nRemaining NaNs after Category-Specific Imputation: {df_retail.isna().sum().sum()}")

### 3. Feature Engineering

In [ ]:
# Derived Business Metrics
df_retail['Order_Month'] = df_retail['Order_Date'].dt.month_name()
df_retail['Profit_Margin_%'] = ((df_retail['Profit'] / df_retail['Sales']) * 100).round(2)
df_retail['Net_Revenue'] = df_retail['Sales'] * (1 - df_retail['Discount'])
df_retail['High_Profit_Flag'] = df_retail['Profit_Margin_%'] >= 15.0

print("Engineered Retail Features:")
print(df_retail[['Order_ID', 'Order_Month', 'Sales', 'Profit', 'Profit_Margin_%', 'High_Profit_Flag']].head())

### 4. Regional & Category Performance Analysis

In [ ]:
regional_kpis = df_retail.groupby(['Region', 'Category']).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Margin=('Profit_Margin_%', 'mean'),
    Order_Count=('Order_ID', 'count')
).round(2)

print("Regional Performance Breakdown:")
print(regional_kpis)

### 5. Multidimensional Profitability Pivot Table

In [ ]:
profit_pivot = pd.pivot_table(
    df_retail,
    values='Profit',
    index='Category',
    columns='Region',
    aggfunc='sum',
    margins=True,
    margins_name='Total_Profit'
).round(2)

print("Regional Profitability Matrix:")
print(profit_pivot)

### 6. Customer Lifetime Value (CLV) & Top Performers

In [ ]:
clv = df_retail.groupby('Customer_ID').agg(
    Orders=('Order_ID', 'nunique'),
    Total_Spend=('Sales', 'sum'),
    Total_Profit_Generated=('Profit', 'sum')
).sort_values(by='Total_Spend', ascending=False)

print("Top 5 Most Valuable Customers:")
print(clv.head(5))

### 7. Summary & KPI Export

In [ ]:
summary_kpis = {
    'Total_Revenue_USD': round(df_retail['Sales'].sum(), 2),
    'Total_Profit_USD': round(df_retail['Profit'].sum(), 2),
    'Overall_Profit_Margin_%': round((df_retail['Profit'].sum() / df_retail['Sales'].sum()) * 100, 2),
    'Total_Unique_Customers': df_retail['Customer_ID'].nunique(),
    'Total_Orders_Processed': len(df_retail)
}

print("=== EXECUTIVE SUMMARY DASHBOARD ===")
for k, v in summary_kpis.items():
    print(f"{k:30}: {v}")